# 08 · The PEFT family map, and BitFit

**Recap (01–07):** LoRA froze the giant `W` and learned a small low-rank change
`B·A`; QLoRA stored that frozen `W` in 4 bits. That whole approach is **one family**
of cheap fine-tuning. There are two others. This notebook maps all three, then
teaches the simplest method of all — **BitFit**.

In [2]:
import numpy as np
rng = np.random.default_rng(0)

## The three families of PEFT (parameter-efficient fine-tuning)

Every method shares one goal: **keep almost all the model frozen, train very few
numbers.** They differ in *how*:

| family | idea | example |
|---|---|---|
| **Reparameterization** | add a small low-rank *change* to existing weights | LoRA / QLoRA ✅ |
| **Selective** | add nothing new — just train a tiny *subset* of existing weights | **BitFit** |
| **Additive** | bolt on a few *new* knobs and train only those | IA3, prompt, prefix |

We've done reparameterization. Now the selective family.

## The primitive we skipped: the *bias*

Back in notebook 01 a layer was `output = W @ x`. The full truth is:

$$\text{output} = W x + b$$

The extra `b` is the **bias** — one number added to each output neuron, a simple
*offset*. `W` decides how to mix the inputs; `b` shifts each result up or down.

In [3]:
W = np.array([[3., 1],
              [0., 2]])
b = np.array([10., -5])      # one bias per output neuron
x = np.array([2., 8])

print("W @ x      =", W @ x)        # the mix
print("W @ x + b  =", W @ x + b)    # ...then shifted by the bias

W @ x      = [14. 16.]
W @ x + b  = [24. 11.]


## BitFit = train ONLY the biases

**BitFit ("Bias-term Fine-tuning")** freezes every weight matrix `W` and trains
**only the bias vectors `b`.** No new parameters, just a tiny subset of existing
ones — and biases are a vanishingly small slice of a layer:

In [4]:
d = k = 1536
weights = d * k          # the W matrix
biases  = d              # one bias per output
print(f"weights in the layer : {weights:,}")
print(f"biases  in the layer : {biases:,}")
print(f"biases are {biases/(weights+biases):.2%} of the layer -> that's all BitFit trains")

weights in the layer : 2,359,296
biases  in the layer : 1,536
biases are 0.07% of the layer -> that's all BitFit trains


Let's *train* only the bias and confirm `W` never moves. The task: a constant
offset `c` added to the layer's output. Bias alone should nail it.

In [5]:
d = k = 4
W = rng.normal(size=(d, k)); W0 = W.copy()    # frozen weights
b = np.zeros(d)                               # the only trainable thing
xs = rng.normal(size=(6, k))
c = np.array([1., -2, 0.5, 3])                # the task: a fixed offset
ys = np.array([W @ x + c for x in xs])        # targets

lr = 0.1
for _ in range(500):
    grad = sum(2 * ((W @ x + b) - y) for x, y in zip(xs, ys)) / len(xs)
    b -= lr * grad                            # update ONLY b

print("learned b    :", np.round(b, 2), " (the true offset was", c, ")")
print("max|W - W0|  :", np.abs(W - W0).max(), " <- weights frozen, as promised")

learned b    : [ 1.  -2.   0.5  3. ]  (the true offset was [ 1.  -2.   0.5  3. ] )
max|W - W0|  : 0.0  <- weights frozen, as promised


The bias learned the offset exactly, weights untouched. **But notice the limit:**
bias can only *shift* outputs, never change *how inputs are mixed* (that's locked in
the frozen `W`). So BitFit is the cheapest method but the least expressive — great
when a task needs small nudges, weak when it needs real remixing.

## Recap

- Three PEFT families: **reparameterization** (LoRA), **selective** (BitFit),
  **additive** (next notebooks).
- A layer is really `W x + b`; the **bias** `b` is a per-neuron offset.
- **BitFit** freezes all weights, trains only biases (~0.1% of params). Cheapest,
  least expressive.

**BAM!** Next: **`09 — how a transformer reads text`**, the primitive the additive
methods need.